In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
#DEFINE THE TRANSFORM
#use 224x224 and ImageNet normalization for EfficientNet-B0.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PropertyImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        
        img_id = self.data.iloc[idx]['id']
        img_name = os.path.join(self.img_dir, f"{img_id}.jpg")
        
        try:
            image = Image.open(img_name).convert('RGB')
        except FileNotFoundError:
            
            image = Image.new('RGB', (400, 400), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        return image, img_id

In [2]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class VisionFeatureExtractor(nn.Module):
    def __init__(self, output_dim=512): # Reduced to 512
        super(VisionFeatureExtractor, self).__init__()
        
        weights = EfficientNet_B0_Weights.DEFAULT
        base_model = efficientnet_b0(weights=weights)
        self.feature_extractor = base_model.features
        self.avgpool = base_model.avgpool
        self.compressor = nn.Linear(1280, output_dim)
        
    def forward(self, x):
        with torch.no_grad():
            x = self.feature_extractor(x)
            x = self.avgpool(x)
            x = torch.flatten(x, 1) # Output: [Batch, 1280]
        
        embeddings = self.compressor(x)
        return embeddings

In [3]:

# Setup Device 
device = torch.device("cpu")
print(f"Using device: {device} ")

# Dataset & Loader
dataset = PropertyImageDataset(csv_file='../data/train_processed.csv', 
                               img_dir='../data/property_images/', 
                               transform=transform)

loader = DataLoader(dataset, 
                    batch_size=16, 
                    shuffle=False, 
                    num_workers=0) 

model = VisionFeatureExtractor(output_dim=512).to(device)
model.eval()


all_embeddings = []
all_ids = []

print("  Extracting features... ")
with torch.no_grad():
    for i, (imgs, ids) in enumerate(tqdm(loader)):
        imgs = imgs.to(device)
        embeddings = model(imgs)
        all_embeddings.append(embeddings.numpy())
        all_ids.extend(ids.numpy())
        
        if i % 100 == 0 and i > 0:
            temp_feat = np.vstack(all_embeddings)
            np.save("../data/image_features_partial.npy", temp_feat)

# 4. Final Save
embedding_matrix = np.vstack(all_embeddings)
np.save("../data/image_features.npy", embedding_matrix)
print(f" Completed! Saved features of shape: {embedding_matrix.shape}")

Using device: cpu 
  Extracting features... 


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1013/1013 [16:16<00:00,  1.04it/s]

 Completed! Saved features of shape: (16201, 512)
